# Stress-constrained topology optimization

This notebook reproduces the L-shape beam example in Section 5.1 of Yang
et al. (2018). The target is Fig. 8: minimum-volume design using the
STM-corrected K-S maximum-stress measure.

> Yang, D., Liu, H., Zhang, W., and Li, S. "Stress-constrained topology
> optimization based on maximum stress measures." *Computers & Structures*
> 198 (2018): 23-39.

In [ ]:
import jax
import jax.numpy as np

jax.config.update("jax_enable_x64", True)

import io
from contextlib import redirect_stdout
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as onp
from IPython.display import Image as DisplayImage
from IPython.display import display
from jax.scipy.special import logsumexp
from jax_fem import logger
from jax_fem.generate_mesh import Mesh, get_meshio_cell_type, rectangle_mesh
from jax_fem.solver import ad_wrapper
from PIL import Image as PILImage

from topax.optimizer import MMA
from topax.problem import TopOptProblem

logger.setLevel("WARNING")

## 1. L-shape finite-element model

In [ ]:
def create_lshape_mesh(nel=100):
    """Create the L-shape mesh used in Fig. 3(b).

    Parameters
    ----------
    nel : int
        Number of elements along each side of the enclosing square.

    Returns
    -------
    Mesh
        L-shape mesh with the unused nodes removed.
    """
    cell_type = get_meshio_cell_type('QUAD4')
    full_mesh = rectangle_mesh(
        Nx=nel,
        Ny=nel,
        domain_x=1.0,
        domain_y=1.0,
    )
    points = onp.asarray(full_mesh.points)
    cells = onp.asarray(full_mesh.cells_dict[cell_type])
    centroids = onp.mean(points[cells], axis=1)
    keep = ~((centroids[:, 0] > 0.4) & (centroids[:, 1] > 0.4))
    cells = cells[keep]

    used_nodes = onp.unique(cells)
    node_map = -onp.ones(len(points), dtype=onp.int32)
    node_map[used_nodes] = onp.arange(len(used_nodes), dtype=onp.int32)
    return Mesh(points[used_nodes], node_map[cells])


class LShapeStress(TopOptProblem):
    """Plane-stress L-shape problem with nodal density variables."""

    def custom_init(self):
        self.fe = self.fes[0]

    def get_tensor_map(self):
        def stress(u_grad, density, penal):
            nu = 0.3
            mu = 1.0 / (2.0 * (1.0 + nu))
            lam = nu / ((1.0 + nu) * (1.0 - 2.0 * nu))
            lam = 2.0 * mu * lam / (lam + 2.0 * mu)
            strain = 0.5 * (u_grad + u_grad.T)
            solid_stress = (
                lam * np.trace(strain) * np.eye(2) + 2.0 * mu * strain
            )
            return density[0] ** penal[0] * solid_stress

        return stress

    def set_params(self, params):
        nodal_density, penal = params
        cell_density = np.mean(nodal_density[self.fe.cells], axis=1)
        self.internal_vars = [
            np.repeat(
                cell_density[:, None, :],
                self.fe.num_quads,
                axis=1,
            ),
            penal * np.ones((self.fe.num_cells, self.fe.num_quads, 1)),
        ]


def center_gradients(points, cells):
    """Return physical shape-function gradients at Q4 element centers."""
    grads_ref = onp.array([
        [-0.25, -0.25],
        [0.25, -0.25],
        [0.25, 0.25],
        [-0.25, 0.25],
    ])
    cell_points = points[cells]
    jacobian = onp.einsum('cni,nj->cij', cell_points, grads_ref)
    return onp.einsum('ni,cij->cnj', grads_ref, onp.linalg.inv(jacobian))


mesh = create_lshape_mesh(nel=100)
points = onp.asarray(mesh.points)
cells = onp.asarray(mesh.cells)


def fixed_location(point):
    return np.logical_and(
        np.isclose(point[1], 1.0, atol=1e-8),
        point[0] <= 0.4 + 1e-8,
    )


problem = LShapeStress(
    mesh,
    vec=2,
    dim=2,
    ele_type='QUAD4',
    dirichlet_bc_info=[
        [fixed_location, fixed_location],
        [0, 1],
        [lambda point: 0.0, lambda point: 0.0],
    ],
)

load_nodes = onp.where(
    onp.isclose(points[:, 0], 1.0)
    & (points[:, 1] >= 0.35 - 1e-8)
    & (points[:, 1] <= 0.4 + 1e-8)
)[0]
nodal_load = onp.zeros((len(points), 2))
nodal_load[load_nodes, 1] = -1.0 / len(load_nodes)
problem.add_nodal_load(np.asarray(nodal_load))

fwd_pred = ad_wrapper(
    problem,
    solver_options={'linear': True, 'spsolve_solver': {}},
    adjoint_solver_options={'spsolve_solver': {}},
)
center_shape_grads = np.asarray(center_gradients(points, cells))

print(f'{len(points)} nodes, {len(cells)} QUAD4 elements')
print(f'Loaded nodes: {len(load_nodes)}, total load: {nodal_load.sum(axis=0)}')

## 2. Element-center von Mises stress

The paper evaluates one stress value at each element center and uses the
relaxed stress interpolation

$$
\boldsymbol{\sigma}_e = \rho_e^{0.5}\mathbf{D}_0\mathbf{B}_c\mathbf{u}_e.
$$

In [ ]:
rho_min = 1e-3
stress_relaxation = 0.5
stress_limit = 70.0
ks_parameter = 8.0


def physical_nodal_density(design):
    return rho_min + (1.0 - rho_min) * design


def element_density(nodal_density):
    return np.mean(nodal_density[cells], axis=1)[:, 0]


def von_mises_stress(displacement, density):
    cell_displacement = displacement[cells]
    displacement_gradient = np.einsum(
        'cnv,cnd->cvd',
        cell_displacement,
        center_shape_grads,
    )
    strain = 0.5 * (
        displacement_gradient + np.swapaxes(displacement_gradient, 1, 2)
    )

    nu = 0.3
    mu = 1.0 / (2.0 * (1.0 + nu))
    lam = nu / ((1.0 + nu) * (1.0 - 2.0 * nu))
    lam = 2.0 * mu * lam / (lam + 2.0 * mu)
    solid_stress = (
        lam * np.trace(strain, axis1=1, axis2=2)[:, None, None] * np.eye(2)
        + 2.0 * mu * strain
    )
    stress = density[:, None, None] ** stress_relaxation * solid_stress

    sigma_x = stress[:, 0, 0]
    sigma_y = stress[:, 1, 1]
    tau_xy = stress[:, 0, 1]
    return np.sqrt(
        sigma_x**2
        - sigma_x * sigma_y
        + sigma_y**2
        + 3.0 * tau_xy**2
        + 1e-20
    )


def analyze_design(design, penal):
    nodal_density = physical_nodal_density(design)
    density = element_density(nodal_density)
    displacement = fwd_pred((nodal_density, penal))[0]
    stress = von_mises_stress(displacement, density)
    ks_stress = logsumexp(
        ks_parameter * stress / stress_limit
    ) / ks_parameter
    return ks_stress, (np.max(stress), density, stress)

## 3. Validation against Fig. 3(b)

In [ ]:
solid_design = np.ones((len(points), 1))
_, (initial_max_stress, _, initial_stress) = analyze_design(
    solid_design,
    np.array(3.0),
)
print(
    f'Initial maximum stress: {float(initial_max_stress):.4f} '
    '(Fig. 3(b): 77.7)'
)

## 4. STM-corrected K-S stress optimization

In [ ]:
def volume_percentage(design):
    density = element_density(physical_nodal_density(design))
    return 100.0 * np.mean(density)


def run_optimization(max_iterations=300, frame_stride=2):
    design = ((0.5 - rho_min) / (1.0 - rho_min)) * np.ones(
        (len(points), 1)
    )
    objective_grad = jax.grad(volume_percentage)(design)
    optimizer = MMA(move=0.1, c_penalty=1e6, d_penalty=0.0)

    penal = 1.0
    correction = 1.0
    records = []
    frames = []

    for iteration in range(1, max_iterations + 1):
        with redirect_stdout(io.StringIO()):
            (
                ks_stress,
                (maximum_stress, density, stress),
            ), ks_grad = jax.value_and_grad(
                analyze_design,
                has_aux=True,
            )(design, np.array(penal))

        target_correction = (maximum_stress / stress_limit) / ks_stress
        correction = 0.5 * float(target_correction) + 0.5 * correction
        objective = volume_percentage(design)
        constraint = correction * ks_stress - 1.0

        record = {
            'iteration': iteration,
            'volume': float(objective) / 100.0,
            'maximum_stress': float(maximum_stress),
            'ks_stress': float(ks_stress),
            'correction': correction,
            'constraint': float(constraint),
            'penal': penal,
        }
        records.append(record)

        if (
            iteration == 1
            or iteration % frame_stride == 0
            or iteration in (10, 100, 200, max_iterations)
        ):
            frames.append({
                **record,
                'density': onp.asarray(density),
                'stress': onp.asarray(stress),
            })

        print(
            f"It.:{iteration:4d}, V/V0:{record['volume']:7.4f}, "
            f"smax:{record['maximum_stress']:8.3f}, "
            f"g:{record['constraint']:+9.3e}, p:{penal:3.1f}"
        )

        if iteration == max_iterations:
            break

        design = np.asarray(
            optimizer.update(
                design,
                objective,
                objective_grad,
                constraint,
                correction * ks_grad,
            )
        )
        penal = min(penal + 0.2, 3.0)

    return {
        'design': onp.asarray(design),
        'records': records,
        'frames': frames,
    }


result = run_optimization()

## 5. Save optimization data

In [ ]:
def cell_field_to_grid(values):
    grid = onp.full((100, 100), onp.nan)
    centroids = onp.mean(points[cells], axis=1)
    ix = onp.floor(100 * centroids[:, 0]).astype(int)
    iy = onp.floor(100 * centroids[:, 1]).astype(int)
    grid[iy, ix] = onp.asarray(values)
    return grid


output_data = Path('docs/data/example_topopt_stress.npz')
output_data.parent.mkdir(parents=True, exist_ok=True)
frames = result['frames']
records = result['records']
onp.savez_compressed(
    output_data,
    points=points,
    cells=cells,
    load_nodes=load_nodes,
    design=result['design'],
    initial_max_stress=float(initial_max_stress),
    frame_iteration=onp.array([frame['iteration'] for frame in frames]),
    frame_density=onp.stack([frame['density'] for frame in frames]),
    frame_stress=onp.stack([frame['stress'] for frame in frames]),
    record_iteration=onp.array([record['iteration'] for record in records]),
    record_volume=onp.array([record['volume'] for record in records]),
    record_maximum_stress=onp.array(
        [record['maximum_stress'] for record in records]
    ),
    record_ks_stress=onp.array([record['ks_stress'] for record in records]),
    record_correction=onp.array(
        [record['correction'] for record in records]
    ),
    record_constraint=onp.array([record['constraint'] for record in records]),
    record_penal=onp.array([record['penal'] for record in records]),
)


def plot_field(ax, values, cmap, vmin, vmax):
    image = ax.imshow(
        cell_field_to_grid(values),
        origin='lower',
        extent=(0.0, 1.0, 0.0, 1.0),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        interpolation='none',
    )
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)
    ax.set_aspect('equal')
    ax.set_axis_off()
    return image

## 6. Optimization animation

In [ ]:
def render_frame(frame):
    with plt.rc_context({
        'text.usetex': True,
        'font.family': 'serif',
        'font.size': 20,
    }):
        fig = plt.figure(figsize=(10, 5), dpi=150)
        density_axis = fig.add_axes([0.035, 0.15, 0.405, 0.70])
        stress_axis = fig.add_axes([0.485, 0.15, 0.405, 0.70])
        colorbar_axis = fig.add_axes([0.925, 0.20, 0.020, 0.60])
        plot_field(
            density_axis, frame['density'], 'gray_r', 0.0, 1.0
        )
        stress_image = plot_field(
            stress_axis,
            frame['stress'],
            'turbo',
            0.0,
            stress_limit,
        )
        fig.text(
            0.2375, 0.075, 'Density', ha='center', va='center', fontsize=23
        )
        fig.text(
            0.6875,
            0.075,
            r'$\sigma_{\mathrm{VM}}$',
            ha='center',
            va='center',
            fontsize=23,
        )
        colorbar = fig.colorbar(
            stress_image,
            cax=colorbar_axis,
            orientation='vertical',
        )
        colorbar.ax.tick_params(labelsize=18)
        fig.suptitle(
            rf"$i={frame['iteration']}\quad "
            rf"V/V_0={100.0 * frame['volume']:.1f}\%\quad "
            rf"\sigma_{{\max}}={frame['maximum_stress']:.1f}\quad "
            rf"p={frame['penal']:.1f}$",
            y=0.965,
            fontsize=22,
        )
        fig.canvas.draw()
        rgba = onp.asarray(fig.canvas.buffer_rgba()).copy()
        plt.close(fig)
    return PILImage.fromarray(rgba)


images = [render_frame(frame) for frame in frames]
gif_path = Path('docs/imgs/example_topopt_stress.gif')
images[0].save(
    gif_path,
    save_all=True,
    append_images=images[1:],
    duration=120,
    loop=0,
    optimize=False,
)
display(DisplayImage(filename=str(gif_path)))

final = records[-1]
print(
    f"Final volume ratio: {100.0 * final['volume']:.2f}% "
    '(Fig. 8: 26.5%)'
)
print(
    f"Final maximum stress: {final['maximum_stress']:.3f} "
    '(Fig. 8: 69.9)'
)